# Explorative Analyse: Arbeitsmarkt Regensburg (Jobcenter) — Teil 1: Exploration

**Datenquelle:** Statistik der Bundesagentur für Arbeit, Bericht *"Eckwerte für Jobcenter"*, Jobcenter Regensburg
(Statistik-Nr. t73906-0), monatliche Excel-Berichte aus `Data/RegensburgJCData/`.

**Zeitraum:** April 2025 – Juni 2026 (15 Berichtsmonate, ein `.xlsx` je Monat).

Jeder Bericht enthält u. a. ein Tabellenblatt **"1.1 Eckwerte"** mit dem aktuellen Monatswert je Merkmal,
getrennt nach *"Insgesamt (SGB II und SGB III)"* und *"Rechtskreis SGB II"* (= Jobcenter-Zuständigkeit).

Dieses Notebook kümmert sich um **Einlesen, Bereinigen und Aufbereiten** der Daten und exportiert am Ende
zwei aufbereitete CSV-Dateien. Die eigentliche **Visualisierung und Auswertung** erfolgt in
[`02_Analyse.ipynb`](02_Analyse.ipynb).

In [1]:
import re
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd

DATA_DIR = Path("..") / "Data" / "RegensburgJCData"
PROCESSED_DIR = Path("..") / "Data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

## 1. Daten einlesen

Alle Monatsberichte parsen und in ein *tidy* DataFrame (`period`, `kategorie`, `merkmal`, `insgesamt`, `sgb2`) überführen.

In [2]:
def normalize_label(label: str) -> str:
    """Entfernt Fußnotenmarker wie ' 1)' oder ' 2) 4)' aus Zeilenbeschriftungen."""
    return re.sub(r"\s*\d\)", "", label).strip()


def parse_eckwerte(path: Path) -> list[dict]:
    """Liest das Blatt '1.1 Eckwerte' eines Monatsberichts aus.

    Spalte B enthält den Wert 'Insgesamt (SGB II und SGB III)', Spalte G den
    Wert für den 'Rechtskreis SGB II' (= Jobcenter) im aktuellen Berichtsmonat.
    Zeilen ohne Zahlwert sind Kategorie-Überschriften (z.B. 'Arbeitslose').
    """
    match = re.search(r"(\d{6})-xlsx\.xlsx$", path.name)
    period = pd.Period(match.group(1), freq="M")

    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb["1.1 Eckwerte"]

    records = []
    category = None
    for row in range(9, 48):
        label = ws.cell(row=row, column=1).value
        insgesamt = ws.cell(row=row, column=2).value
        sgb2 = ws.cell(row=row, column=7).value

        if not isinstance(label, str) or not label.strip():
            continue
        label = normalize_label(label)

        is_value_row = isinstance(insgesamt, (int, float)) or isinstance(sgb2, (int, float))
        if not is_value_row:
            category = label
            continue

        records.append({
            "period": period,
            "kategorie": category,
            "merkmal": label,
            "insgesamt": insgesamt,
            "sgb2": sgb2,
        })
    return records


files = sorted(DATA_DIR.glob("*.xlsx"))
print(f"{len(files)} Berichte gefunden: {files[0].name} ... {files[-1].name}")

all_records = [rec for f in files for rec in parse_eckwerte(f)]
df = pd.DataFrame(all_records).sort_values(["period", "kategorie", "merkmal"]).reset_index(drop=True)
df["jahr"] = df["period"].dt.year
df["monat"] = df["period"].dt.month
df["label"] = df["kategorie"] + " | " + df["merkmal"]
df.head(10)

15 Berichte gefunden: jc-eckwerte-t73906-0-202504-xlsx.xlsx ... jc-eckwerte-t73906-0-202606-xlsx.xlsx


C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:33: UserWarning: DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.
  warn("DrawingML support is incomplete and limited to charts and images only. Shapes and drawings will be lost.")
C:\Users\funke\AppData\Roaming\Python\Python314\site-packages\openpyxl\reader\drawings.py:67: UserWarning: wmf image format is not supported so the image is being dropped
  warn(msg)


,period,kategorie,merkmal,insgesamt,sgb2,jahr,monat,label
0,2025-04,Arbeitslose,15 bis unter 25 Jahre,355.0,160.0,2025,4,Arbeitslose | 15 bis unter 25 Jahre
1,2025-04,Arbeitslose,25 bis unter 50 Jahre,1833.0,854.0,2025,4,Arbeitslose | 25 bis unter 50 Jahre
2,2025-04,Arbeitslose,50 Jahre und älter,1306.0,443.0,2025,4,Arbeitslose | 50 Jahre und älter
3,2025-04,Arbeitslose,55 Jahre und älter,1008.0,302.0,2025,4,Arbeitslose | 55 Jahre und älter
4,2025-04,Arbeitslose,Abgang (12-Monatssumme),9024.0,3348.0,2025,4,Arbeitslose | Abgang (12-Monatssumme)
5,2025-04,Arbeitslose,Abgang (im Monat),864.0,250.0,2025,4,Arbeitslose | Abgang (im Monat)
6,2025-04,Arbeitslose,Arbeitslosenquote,3.0,1.3,2025,4,Arbeitslose | Arbeitslosenquote
7,2025-04,Arbeitslose,Ausländer,1216.0,735.0,2025,4,Arbeitslose | Ausländer
8,2025-04,Arbeitslose,Bestand,3494.0,1457.0,2025,4,Arbeitslose | Bestand
9,2025-04,Arbeitslose,Frauen,1507.0,667.0,2025,4,Arbeitslose | Frauen


## 2. Datenüberblick und Bereinigung

In [3]:
print("Shape:", df.shape)
print("Berichtsmonate:", df['period'].min(), "-", df['period'].max(), f"({df['period'].nunique()} Monate)")
print("Duplikate:", df.duplicated().sum())
print("Fehlende Werte (sgb2):", df['sgb2'].isna().sum(), "von", len(df))
print("\nMerkmale je Kategorie:")
for kat, sub in df.groupby("kategorie", sort=False):
    print(f"- {kat}: {sorted(sub['merkmal'].unique())}")

Shape: (465, 8)
Berichtsmonate: 2025-04 - 2026-06 (15 Monate)
Duplikate: 0
Fehlende Werte (sgb2): 90 von 465

Merkmale je Kategorie:
- Arbeitslose: ['15 bis unter 25 Jahre', '25 bis unter 50 Jahre', '50 Jahre und älter', '55 Jahre und älter', 'Abgang (12-Monatssumme)', 'Abgang (im Monat)', 'Arbeitslosenquote', 'Ausländer', 'Bestand', 'Frauen', 'Langzeitarbeitslose', 'Männer', 'Zugang (12-Monatssumme)', 'Zugang (im Monat)', 'schwerbehinderte Menschen']
- Arbeitsuchende: ['Bestand']
- Grundsicherung für Arbeitsuchende: ['Bedarfsgemeinschaften (BG)', 'Personen in Bedarfsgemeinschaften (PERS)', 'dar. Regelleistungsberechtigte (RLB)', 'dav. erwerbsfähige Leistungsberechtigte (ELB)', 'nicht erwerbsfähige Leistungsberechtigte (NEF)']
- Unterbeschäftigung: ['Arbeitslosigkeit im weiteren Sinne', 'Unterbeschäftigung (ohne Kurzarbeit)', 'Unterbeschäftigung im engeren Sinne', 'Unterbeschäftigungsquote']
- gemeldete Arbeitsstellen: ['Bestand', 'Zugang (12-Monatssumme)', 'Zugang (im Monat)', 'sozial

In [4]:
print("Kennzahlen der SGB-II-Werte (deskriptive Statistik):")
df["sgb2"].describe()

Kennzahlen der SGB-II-Werte (deskriptive Statistik):


count     375.000000
mean     1593.675741
std      1499.720374
min         1.200000
25%       335.500000
50%       869.000000
75%      2665.938548
max      5421.148834
Name: sgb2, dtype: float64

## 3. Kennzahlen-Zeitreihe aufbereiten

Die für die Analyse relevanten Merkmale werden aus dem langen Format per `pivot_table()` in eine breite
Zeitreihen-Tabelle (eine Zeile je Monat, eine Spalte je Kennzahl) überführt.

In [5]:
kennzahlen_labels = {
    "Arbeitslose | Bestand": "Arbeitslose (Bestand)",
    "Arbeitslose | Arbeitslosenquote": "Arbeitslosenquote (%)",
    "Arbeitslose | Langzeitarbeitslose": "Langzeitarbeitslose",
    "Unterbeschäftigung | Unterbeschäftigungsquote": "Unterbeschäftigungsquote (%)",
    "Grundsicherung für Arbeitsuchende | Bedarfsgemeinschaften (BG)": "Bedarfsgemeinschaften",
    "Grundsicherung für Arbeitsuchende | Personen in Bedarfsgemeinschaften (PERS)": "Personen in BG",
    "Grundsicherung für Arbeitsuchende | dar. Regelleistungsberechtigte (RLB)": "Regelleistungsberechtigte (RLB)",
    "Grundsicherung für Arbeitsuchende | dav. erwerbsfähige Leistungsberechtigte (ELB)": "Erwerbsfähige Leistungsberechtigte (ELB)",
    "Grundsicherung für Arbeitsuchende | nicht erwerbsfähige Leistungsberechtigte (NEF)": "Nicht erwerbsfähige Leistungsberechtigte (NEF)",
}

subset = df[df["label"].isin(kennzahlen_labels)].copy()
subset["kennzahl"] = subset["label"].map(kennzahlen_labels)

kennzahlen = subset.pivot_table(index="period", columns="kennzahl", values="sgb2")
kennzahlen = kennzahlen[list(kennzahlen_labels.values())]
kennzahlen.index = kennzahlen.index.to_timestamp()

print("Mittelwert / Std je Kennzahl (NumPy):")
for spalte in kennzahlen.columns:
    werte = kennzahlen[spalte].to_numpy()
    print(f"- {spalte}: mean={np.mean(werte):.1f}, std={np.std(werte):.1f}")

kennzahlen

Mittelwert / Std je Kennzahl (NumPy):
- Arbeitslose (Bestand): mean=1520.4, std=44.6
- Arbeitslosenquote (%): mean=1.3, std=0.0
- Langzeitarbeitslose: mean=621.5, std=58.7
- Unterbeschäftigungsquote (%): mean=1.7, std=0.0
- Bedarfsgemeinschaften: mean=2674.4, std=40.8
- Personen in BG: mean=5203.7, std=82.9
- Regelleistungsberechtigte (RLB): mean=4905.4, std=79.0
- Erwerbsfähige Leistungsberechtigte (ELB): mean=3555.5, std=55.6
- Nicht erwerbsfähige Leistungsberechtigte (NEF): mean=1349.9, std=28.0


kennzahl,Arbeitslose (Bestand),Arbeitslosenquote (%),Langzeitarbeitslose,Unterbeschäftigungsquote (%),Bedarfsgemeinschaften,Personen in BG,Regelleistungsberechtigte (RLB),Erwerbsfähige Leistungsberechtigte (ELB),Nicht erwerbsfähige Leistungsberechtigte (NEF)
period,,,,,,,,,
2025-04-01,1457.0,1.3,526.0,1.8,2766.959069,5421.148834,5122.424563,3701.239670,1421.184893
2025-05-01,1438.0,1.2,558.0,1.8,2707.556182,5295.629265,4989.629265,3612.418872,1377.210393
2025-06-01,1509.0,1.3,551.0,1.8,2696.605759,5275.539536,4960.102098,3584.217725,1375.884374
2025-07-01,1475.0,1.3,561.0,1.8,2616.920054,5092.808598,4788.461509,3477.798107,1310.663402
2025-08-01,1587.0,1.4,585.0,1.8,2688.060659,5267.867812,4961.929122,3593.655245,1368.273878
2025-09-01,1585.0,1.4,592.0,1.8,2669.283813,5218.350099,4934.149358,3564.447439,1369.701919
2025-10-01,1519.0,1.3,606.0,1.7,2629.348039,5120.198766,4847.485769,3499.340443,1348.145325
2025-11-01,1517.0,1.3,623.0,1.7,2632.776815,5126.678771,4839.956063,3504.915856,1335.040207
2025-12-01,1466.0,1.3,610.0,1.7,2619.193501,5137.270730,4838.074529,3489.353039,1348.721490


## 4. Export für die Analyse

Beide DataFrames werden als CSV abgelegt, damit `02_Analyse.ipynb` unabhängig davon starten kann.

In [6]:
df.to_csv(PROCESSED_DIR / "eckwerte_long.csv", index=False)
kennzahlen.to_csv(PROCESSED_DIR / "kennzahlen_sgb2.csv", index_label="period")
print("Exportiert nach:", PROCESSED_DIR.resolve())

Exportiert nach: C:\Users\funke\OneDrive\Skills_Update\arbeitsmarkt-regensburg\Data\processed


Weiter geht es in [`02_Analyse.ipynb`](02_Analyse.ipynb) mit den Visualisierungen und der inhaltlichen Auswertung.